# NLP Lab 2 Assignment: Sentence Generation using N-Grams

In this assignment, we will build a simple **N-gram language model** from scratch and use it to generate **whole sentences** word by word.

**Steps we will follow:**
1. Read the text from `corpus.txt`
2. Clean the text and split it into sentences
3. Tokenize each sentence into words and add sentence boundary markers (`<s>` and `</s>`)
4. Build an N-gram model (Bigram / Trigram)
5. Generate whole sentences one word at a time using the model

## Step 1: Read the corpus

We open the `corpus.txt` file and read all the text into a single string variable `text`.

In [92]:
# Open the corpus file and read all the text
with open("corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Show the length of the text and a small preview
print("Total characters in corpus:", len(text))
print("First 200 characters:")
print(text[:500])

Total characters in corpus: 124155
First 200 characters:
=== ০১. আপনি কি ভূত দেখেছেন ===

‘আপনি কি ভূত দেখেছেন স্যার? ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……..’

মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না। কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না। প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে। কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়। মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির। ভদ্রলোক মধ্যবয়স্ক। গোলাকার মুখে পুরুষ্ট গোঁফ। কুস্তিগির-কুস্তিগির 


## Step 2: Clean and split into sentences

We split the big text into a list of sentences using simple punctuation marks (`.`, `?`, `!`, `।`).
We then strip extra whitespace from each sentence.

In [93]:
import re

# Replace newlines with spaces so the text becomes one long line
text = text.replace("\n", " ")

# Remove Bengali numerals (০ ১ ২ ৩ ৪ ৫ ৬ ৭ ৮ ৯) and English digits (0-9)
text = re.sub(r'[০-৯0-9]', '', text)

# Remove '=' characters
text = text.replace('=', '')

# Remove right single quotation mark (’) and left single quotation mark (‘)
text = text.replace('’', '').replace('‘', '')

# Split the text into sentences using sentence-end punctuation:
# '।'             -> Bengali full stop — equivalent to '.'

raw_sentences = re.split(r'[.!?।]+', text)

# Remove empty strings and strip extra spaces
sentences = []
for s in raw_sentences:
    s = s.strip()
    if len(s) > 0:
        sentences.append(s)

# Show how many sentences we got
print("Total sentences:", len(sentences))
print("First 15 sentences:")
for s in sentences[:20]:
    print(" ", s)

# Write the cleaned sentences to a new file
# One sentence per line so it is easy to read and reuse later
with open("cleaned-corpus.txt", "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s + "\n")

print("\nCleaned corpus written to cleaned-corpus.txt")
print("Total lines written:", len(sentences))

Total sentences: 3217
First 15 sentences:
  আপনি কি ভূত দেখেছেন   আপনি কি ভূত দেখেছেন স্যার
  ইংরেজিতে যাকে বলে spirit, ghost, astral body মানে প্রেতাত্মার কথা বলছি, অশরীরী……
  মিসির আলি প্রশ্নটির জবাব দেবেন কি না বুঝতে পারছেন না
  কিছু মানুষ আছে যারা প্রশ্ন করে, কিন্তু জবাব শুনতে চায় না
  প্রশ্ন করেই হড়বড় করে কথা বলতে থাকে
  কথার ফাঁকে-ফাঁকে আবার প্রশ্ন করে, আবার নিজেই জবাব দেয়
  মিসির আলির কাছে মনে হচ্ছে তাঁর সামনের চেয়ারে বসে থাকা এই মানুষটি সেই প্রকৃতির
  ভদ্রলোক মধ্যবয়স্ক
  গোলাকার মুখে পুরুষ্ট গোঁফ
  কুস্তিগির-কুস্তিগির চেহারা
  কথার মাঝখানে হাসার অভ্যাস আছে
  হাসার সময় কোনো শব্দ হয় না, কিন্তু সারা শরীর দুলতে থাকে
  ওসমান গনি নামের এই মানুষটির প্রধান বৈশিষ্ট্য অবশ্য নিঃশব্দে হাসার ক্ষমতা নয়; প্রধান বৈশিষ্ট্য হচ্ছে তাঁর নিচের পাটির একটি এবং ওপরের পাটির দুটি দাঁত সোনা দিয়ে বাঁধানো
  যে-যুগে রুট ক্যানালিং-এর মতো আধুনিক দন্ত চিকিৎসা শুরু হয়েছে, সে-যুগে কেউ সোনা দিয়ে দাঁত বাঁধায় না
  এই ভদ্রলোক বাঁধিয়েছেন
  ধবধবে সাদা দাঁতের মাঝে ঝকঝকে তিনটি সোনালি দাঁত
  কথা বলছেন না কে

## Step 3: Tokenize and add sentence boundary markers

We split each sentence into words (tokens).
Then we add `<s>` at the start and `</s>` at the end of every sentence.

These markers are important because:
* `<s>` tells the model "a new sentence is starting here".
* `</s>` tells the model "the sentence has ended here".

Finally we flatten everything into a single list of tokens called `tokens`.

In [94]:
# Step 3a: Build the vocabulary and decide which words to replace with <unk>.

from collections import Counter

UNK_THRESHOLD = 2   # words appearing fewer than this many times become <unk>

# Count every word across all sentences
word_counts = Counter()
for sentence in sentences:
    for word in sentence.split():
        word_counts[word] += 1

# Build the vocabulary: keep words whose count is >= UNK_THRESHOLD
vocabulary = {w for w, c in word_counts.items() if c >= UNK_THRESHOLD}

# Add the sentence boundary markers and <unk> to the vocabulary explicitly.
# These tokens should always be "known" to the model.
vocabulary.add("<s>")
vocabulary.add("</s>")
vocabulary.add("<unk>")

print("Vocabulary size (V):", len(vocabulary))
print("Total unique raw words before UNK replacement:", len(word_counts))
print("Words replaced by <unk>:",
      sum(1 for c in word_counts.values() if c < UNK_THRESHOLD))


# Step 3b: Tokenize each sentence into words and add sentence boundary
# markers (<s> and </s>), replacing out-of-vocabulary words with <unk>.

# This list will hold all tokens from all sentences
tokens = []

# Go through each sentence one by one
for sentence in sentences:
    # Split the sentence into words using whitespace
    words = sentence.split()

    # Replace any word not in the vocabulary with <unk>
    words = [w if w in vocabulary else "<unk>" for w in words]

    # Only keep sentences that have at least 1 word
    if len(words) > 0:
        # Add start marker, the words, and end marker
        sentence_tokens = ['<s>'] + words + ['</s>']

        # Extend the master token list with these tokens
        tokens.extend(sentence_tokens)

# Show total tokens and a preview
print("Total tokens:", len(tokens))
print("First 30 tokens:")
print(tokens[:100])

Vocabulary size (V): 1915
Total unique raw words before UNK replacement: 4317
Words replaced by <unk>: 2405
Total tokens: 26951
First 30 tokens:
['<s>', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'আপনি', 'কি', 'ভূত', 'দেখেছেন', 'স্যার', '</s>', '<s>', 'ইংরেজিতে', 'যাকে', 'বলে', '<unk>', '<unk>', '<unk>', '<unk>', 'মানে', '<unk>', 'কথা', '<unk>', '<unk>', '</s>', '<s>', 'মিসির', 'আলি', '<unk>', 'জবাব', 'দেবেন', 'কি', 'না', 'বুঝতে', 'পারছেন', 'না', '</s>', '<s>', 'কিছু', 'মানুষ', 'আছে', 'যারা', 'প্রশ্ন', 'করে,', 'কিন্তু', 'জবাব', 'শুনতে', 'চায়', 'না', '</s>', '<s>', 'প্রশ্ন', 'করেই', 'হড়বড়', 'করে', 'কথা', 'বলতে', 'থাকে', '</s>', '<s>', 'কথার', '<unk>', 'আবার', 'প্রশ্ন', 'করে,', 'আবার', 'নিজেই', 'জবাব', 'দেয়', '</s>', '<s>', 'মিসির', 'আলির', 'কাছে', 'মনে', 'হচ্ছে', 'তাঁর', 'সামনের', 'চেয়ারে', 'বসে', 'থাকা', 'এই', 'মানুষটি', 'সেই', 'প্রকৃতির', '</s>', '<s>', 'ভদ্রলোক', 'মধ্যবয়স্ক', '</s>', '<s>', '<unk>', 'মুখে', '<unk>', '<unk>', '</s>', '<s>', '<unk>', 'চেহারা', '</s>']


## Step 4: Build an N-gram model

Here we build an N-gram model manually using a Python dictionary.

The structure is:
```
ngram_model[(w1, w2, ...)] = { next_word: count }
```

For example, for a **Bigram (n=2)** model:
* Key: a tuple of 1 word (the history)
* Value: a dictionary of next words and how many times they appeared

We use a sliding window over the `tokens` list:
```
[w1, w2, w3, w4, w5]
 history = (w1, w2)
 next    = w3
```
Then we shift the window by one step until the end.

In [95]:
# ----- Generic N-gram builder with Laplace (add-1) smoothing support -----

def build_ngram_model(tokens, n):
    """
    Build an N-gram count model from a list of tokens.

    Returns:
        counts        : dict mapping history-tuple -> {next_word: count}
        history_count : dict mapping history-tuple -> total count of that history
        vocab         : sorted list of all words in the vocabulary
        V             : size of the vocabulary (used for Laplace smoothing)
    """
    counts = {}
    history_count = {}

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i : i + n - 1])
        next_word = tokens[i + n - 1]

        # Tally (history, next_word) pairs
        if history not in counts:
            counts[history] = {}
        counts[history][next_word] = counts[history].get(next_word, 0) + 1

        # Tally total count for the history
        history_count[history] = history_count.get(history, 0) + 1

    vocab = sorted(set(tokens))
    V = len(vocab)

    return counts, history_count, vocab, V


def laplace_sample(history, counts, history_count, vocab, V):
    """
    Sample one next word using Laplace (add-1) smoothing.

    P(w | h) = ( C(h, w) + 1 ) / ( C(h) + V )

    """
    observed = counts.get(history, {})
    C_h = history_count.get(history, 0)

    # Build (word, smoothed_probability) pairs for every word in V
    choices = vocab
    weights = [observed.get(w, 0) + 1 for w in vocab]   # +1 = Laplace

    return random.choices(choices, weights=weights)[0]

In [96]:
# ----- Bigram model (n = 2) with Laplace smoothing -----

n = 2
ngram_model, history_count, vocab, V = build_ngram_model(tokens, n)

print("Vocabulary size (V):", V)
print("Total unique bigram histories:", len(ngram_model))

example_history = ('<s>',)
print("\nExample: next words after", example_history, "->")
print(ngram_model.get(example_history, {}))

example_next_words = ngram_model.get(example_history, {})
sorted_next_words = sorted(
    example_next_words.items(),
    key=lambda item: item[1],
    reverse=True
)

print("\nSame next words, sorted by count (descending):")
for word, count in sorted_next_words:
    print(f"  {word!r:5s} -> {count}")

# ----- Bigram model (n = 2) WITHOUT smoothing -----

ngram_model_ns, history_count_ns, vocab_ns, V_ns = build_ngram_model(tokens, n)

print("\nBigram model built. V =", V_ns)

Vocabulary size (V): 1915
Total unique bigram histories: 1915

Example: next words after ('<s>',) ->
{'আপনি': 83, 'ইংরেজিতে': 1, 'মিসির': 213, 'কিছু': 10, 'প্রশ্ন': 2, 'কথার': 3, 'ভদ্রলোক': 4, '<unk>': 320, 'হাসার': 1, 'ওসমান': 39, 'এই': 54, 'ধবধবে': 2, 'কথা': 2, 'জ্বি': 32, 'আমি': 88, 'ঘাম': 1, 'এক': 16, 'পায়ের': 2, 'ডাক্তার': 10, 'ডাক্তারদের': 5, 'তার': 30, 'অ্যান্টি': 2, 'বাইরে': 3, 'দশ': 2, 'ভূতের': 5, 'আমার': 66, 'গত': 6, 'দু': 9, 'যদি': 6, 'খুব': 18, 'আমারো': 1, 'খুবই': 3, 'গল্পটা': 3, 'আচ্ছা': 7, 'আপনার': 65, 'সে': 21, 'বই': 2, 'অনেক': 8, 'তখন': 7, 'যাদের': 3, 'না,': 22, 'প্রথম': 3, 'এখন': 31, 'প্রায়ই': 3, 'জ্বি-না': 6, 'ধরেন': 1, 'এরা': 4, 'ও': 8, 'পাঁচ': 1, 'সব': 9, 'মার': 2, 'একগাদা': 1, 'সন্ধ্যার': 5, 'রোজ': 4, 'শব্দ': 5, 'গলার': 2, 'মূল': 2, 'বাড়ির': 6, 'ছোট্ট': 2, 'ছাদে': 1, 'কাজের': 4, 'রাতে': 9, 'মাথার': 2, 'টেবিল': 1, 'তখনি': 1, 'চোখের': 3, 'দু-এক': 1, 'তারা': 9, 'একজনের': 1, 'কারণ': 19, 'কাজেই': 11, 'ধমক': 1, 'তারপর': 6, 'ছোট': 1, 'জ্বর': 1, 'চলুন': 2, 'ইন্টারেস্টিং

## Step 5: Generate a sentence using the N-gram model

Now we will generate a brand new sentence **one word at a time** using the model.

How it works:
1. Start with the history `( '<s>', )` (sentence start marker).
2. Look at `ngram_model[history]` to see all possible next words and their counts.
3. Pick a next word (we use the most common one for simplicity).
4. Append it to the sentence.
5. Slide the window: drop the oldest word, add the new word.
6. Repeat until we see `</s>` (sentence end marker) or hit a max length.

In [100]:
def generate_sentence_smooth(ngram_model, history_count, vocab, V, max_len=15):
    history = ('<s>',)
    generated = []
    for _ in range(max_len):
        next_word = laplace_sample(history, ngram_model, history_count, vocab, V)
        if next_word == '</s>':
            break
        if next_word == '<unk>':
            continue
        generated.append(next_word)
        history = (next_word,)
    return ' '.join(generated)


def generate_sentence_no_smooth(ngram_model, max_len=15):
    history = ('<s>',)
    generated = []
    for _ in range(max_len):
        if history not in ngram_model:
            break
        next_words = ngram_model[history]
        choices = list(next_words.keys())
        weights = list(next_words.values())
        next_word = random.choices(choices, weights=weights)[0]
        if next_word == '</s>':
            break
        generated.append(next_word)
        history = (next_word,)
    return ' '.join(generated)


# No-smoothing trigram model — built here (not in Step 6) so Step 5
# doesn't depend on Step 6 having run first.
trigram_model_ns, trigram_history_count_ns, trigram_vocab_ns, trigram_V_ns = (
    build_ngram_model(tokens, 3)
)


def generate_sentence_trigram_no_smooth(trigram_model_ns, max_len=15):
    history = ('</s>', '<s>')
    generated = []
    for _ in range(max_len):
        if history not in trigram_model_ns:
            break
        next_words = trigram_model_ns[history]
        choices = list(next_words.keys())
        weights = list(next_words.values())
        next_word = random.choices(choices, weights=weights)[0]
        if next_word == '</s>':
            break
        generated.append(next_word)
        history = (history[1], next_word)
    return ' '.join(generated)


print("=" * 70)
print("BIGRAM — WITHOUT smoothing")
print("=" * 70)
for i in range(3):
    print(f"{i+1}. {generate_sentence_no_smooth(ngram_model_ns, max_len=15)}")

print()
print("=" * 70)
print("BIGRAM — WITH Laplace (add-1) smoothing")
print("=" * 70)
for i in range(3):
    print(f"{i+1}. {generate_sentence_smooth(ngram_model, history_count, vocab, V, max_len=15)}")

print()
print("=" * 70)
print("TRIGRAM — WITHOUT smoothing")
print("=" * 70)
for i in range(3):
    print(f"{i+1}. {generate_sentence_trigram_no_smooth(trigram_model_ns, max_len=15)}")

print()
print("=" * 70)
print("TRIGRAM — WITH Laplace (add-1) smoothing")
print("=" * 70)
for i in range(3):
    print(f"{i+1}. {generate_sentence_trigram(trigram_model, trigram_history_count, trigram_vocab, trigram_V, max_len=15)}")

BIGRAM — WITHOUT smoothing
1. বুদ্ধিমান <unk> আছে
2. ঘটনার <unk> <unk> মেয়ে বলে ভালো লাগে
3. ঘর

BIGRAM — WITH Laplace (add-1) smoothing
1. জ্বি-না, শূন্য রবীন্দ্রনাথ খাবেন অপূর্ব কোনোদিন ব্যাপারটি লাইব্রেরি ফেলল দিয়েছে কথাবার্তা মৃত ব্যবহার যাব, হঠাৎ
2. জ্বি-না পাতার বলতে নি অশরীরী দেখাশোনা শরীরে হবার পড়েছে খবরটি শুনব কনসিভ কতদিন নেই—এই এ-বাড়ির
3. … ড্রেসিং ভিজতে-ভিজতে থাকেন দুটোর বিষয় খোলা ঘুমুতে অনুষ্ঠানে ভালো, গেলেন পোস্ট টাকা বাথরুম ছাড়া

TRIGRAM — WITHOUT smoothing
1. মানুষটি একজন বেহালাবাদক এবং মৃত
2. মিনিস্টার সাহেব মিসির আলির তন্দ্রার মতো হল
3. রাগী রাগী চোখে তিনি চায়ের কাপের দিকে

TRIGRAM — WITH Laplace (add-1) smoothing
1. আব্দুল অতিথি প্রশ্নের অ্যাংলো থেকেও in ক্যাবিনেট কাপে মন্দ সাফাকাত করবেন ভেরি ধনবান বিচার মানুষ
2. মধ্যেই নিয়ন্ত্রিত বারবার ভালো বিছানা কিছুক্ষণের রিকশাওয়ালা অসুখবিসুখ কুঁচকে টাকায় হাস্যকর পড়েছেন বাসায় বলছেন পুড়তে
3. জীবনে বয়সী নাইট্রেটের মধ্যে দু-এক নেন বইয়ে সঙ্গেই মানুষ খেতে—খেতে নেমে খারাপ পুড়তে দিলরুবা জীবনে


## Step 6: Try a Trigram model (n=3)

If we set `n = 3`, the model becomes a **Trigram** model:
* history = tuple of 2 words
* next word = the third word

Trigrams usually produce sentences that flow a little more naturally because the model "remembers" 2 words of context instead of just 1.

**Note:** with a small or noisy corpus, some histories will not exist in the model, so generation may stop early. That is normal.

In [101]:
# Build a trigram model (n=3) with Laplace smoothing
n = 3
trigram_model, trigram_history_count, trigram_vocab, trigram_V = (
    build_ngram_model(tokens, n)
)

print("Total unique trigram histories:", len(trigram_model))

boundary = ('</s>', '<s>')
print(f"Occurrences of history {boundary}:",
      sum(trigram_model.get(boundary, {}).values()))


def generate_sentence_trigram(trigram_model, history_count, vocab, V,
                              max_len=20):
    history = ('</s>', '<s>')
    generated = []
    for _ in range(max_len):
        next_word = laplace_sample(history, trigram_model, history_count,
                                   vocab, V)
        if next_word == '</s>':
            break
        if next_word == '<unk>':
            continue
        generated.append(next_word)
        history = (history[1], next_word)
    return ' '.join(generated)


print("\nGenerated sentences from the trigram model (with Laplace smoothing):\n")
for i in range(5):
    sentence = generate_sentence_trigram(
        trigram_model, trigram_history_count, trigram_vocab, trigram_V,
        max_len=15
    )
    print(f"{i+1}. {sentence}")

Total unique trigram histories: 12323
Occurrences of history ('</s>', '<s>'): 3216

Generated sentences from the trigram model (with Laplace smoothing):

1. কিন্তু পড়েছেন থাকুক আসুন, নিচ্ছি কথায় উপদেশ মুখে হ্যাঁ, মাথা নিয়তি বইটা চাও স্ত্রীর যাই
2. ডাক্তার ট্যাপ ব্যবসায়ী ফিরে অল্পবয়সী এ-ধরনের ঢুকে প্রেতাত্মা কী ঢুকলে কিংবা পাচ্ছেন শেষে খাওয়া খুঁটিয়ে
3. বসে অভিনয় একধরনের নিজ গুরুত্বের তদন্তের সোনার করলেও দিতে স্থান কিছুক্ষণের আচ্ছা নেন লিখে ঘটে
4. এয়ার কোনোদিন গ্রহণ দু চাচা থাকবেই ক্ষমতা রোজগার যথেষ্ট ভয়াবহ তিনটি খেতে—খেতে লাগোয়া হয় ভর্তি
5. বাথরুম বাবা-মা মানুষ, পাওয়ার ইংরেজিতে করল নাকি কিচ্ছু হতে উনিই স্বস্তি সেজে আনলেন একগাদা অবাক
